In [ ]:
from dotenv import load_dotenv
import os
from google import genai
from google.genai import types
import pandas as pd

load_dotenv()

In [ ]:
game_logs = pd.read_excel("C:\\Users\\andre\\BBL_Stats_DATA\\BBL Season 2 Stats.xlsx",sheet_name="Game Logs")
game_logs = game_logs.dropna()

In [ ]:
#Change this for each week to run should aggregate stats accordingly
current_week = 2

#Makes dataframe of only stats from past weeks
past_game_logs = game_logs[game_logs["Week"] < current_week ]

current_week_log = game_logs[game_logs["Week"] == current_week]


In [ ]:
current_week_log['FG_P'] = ((current_week_log['FGM'] / current_week_log['FGA']) * 100).round(2)

current_week_log['3P_P'] = ((current_week_log['3PM'] / current_week_log['3PA']) * 100).round(2)

past_game_logs['FG_P'] = ((past_game_logs['FGM'] / past_game_logs['FGA']) * 100).round(2)

past_game_logs['3P_P'] = ((past_game_logs['3PM'] / past_game_logs['3PA']) * 100).round(2)

In [ ]:
current_week_log


In [ ]:
# Seperate dataframes into each game


#temporary fix this needs and index reset, figure out later
game1_id = current_week_log.at[17,'Game_ID']

game1 = current_week_log[current_week_log['Game_ID'] == game1_id]
game2 = current_week_log[current_week_log['Game_ID'] != game1_id]


In [ ]:
# Groups each by team

game1_team_groups = game1.groupby("Team",sort=False)[["PTS","FGA", "FGM", "3PA", "3PM","REB", "AST", "STL", "BLK"]].sum().reset_index()
game2_team_groups = game2.groupby("Team",sort=False)[["PTS","FGA", "FGM", "3PA", "3PM","REB", "AST", "STL", "BLK"]].sum().reset_index()

game1_team_groups['FG_P'] = ((game1_team_groups['FGM'] / game1_team_groups['FGA']) * 100).round(2)

game1_team_groups['3P_P'] = ((game1_team_groups['3PM'] / game1_team_groups['3PA']) * 100).round(2)

game2_team_groups['FG_P'] = ((game2_team_groups['FGM'] / game2_team_groups['FGA']) * 100).round(2)

game2_team_groups['3P_P'] = ((game2_team_groups['3PM'] / game2_team_groups['3PA']) * 100).round(2)


In [ ]:
team_names =  {
    'WESTDIST' : 'West District Tigers',
    'DAB' : 'Dab Dynasty',
    'THREES' : 'Drilled Threes Burrito',
    'ECTA' : 'Emerald City Throat Artist'
}

In [ ]:
def process_game(game_team_groups):
    winner = game_team_groups.iloc[game_team_groups['PTS'].idxmax()]
    loser = game_team_groups.iloc[game_team_groups['PTS'].idxmin()]
    return f"{team_names[winner["Team"]]} beat {team_names[loser["Team"]]} {winner["PTS"]} to {loser["PTS"]}. {team_names[winner["Team"]]} had {winner['REB']} rebounds, {winner['AST']} assists, {winner['STL']} steals, {winner['BLK']} blocks and shot {winner['FG_P']} percent from the field and shot {winner['3P_P']} percent from three. {team_names[loser["Team"]]} had {loser['REB']} rebounds, {loser['AST']} assists, {loser['STL']} steals, {loser['BLK']} blocks and shot {loser['FG_P']} percent from the field and shot {loser['3P_P']} percent from three."  

In [56]:
game1_string = process_game(game1_team_groups)
game2_string = process_game(game2_team_groups)

game2_string

'Emerald City Throat Artist beat Dab Dynasty 36.0 to 56.0. Emerald City Throat Artist had 15.0 rebounds, 9.0 assists, 3.0 steals, 1.0 blocks and shot 44.44 percent from the field and shot 33.33 percent from three. Dab Dynasty had 16.0 rebounds, 10.0 assists, 3.0 steals, 2.0 blocks and shot 53.49 percent from the field and shot 58.82 percent from three.'

In [ ]:
player_averages = past_game_logs.groupby("Name",sort=False)[["PTS", "FGA", "FGM", "3PA", "3PM", "REB", "AST", "STL", "BLK"]].mean().reset_index()

player_averages['FG_P'] = ((player_averages['FGM'] / player_averages['FGA']) * 100).round(2)

player_averages['3P_P'] = ((player_averages['3PM'] / player_averages['3PA']) * 100).round(2)

player_averages

In [ ]:
lines = []

for _, row in player_averages.iterrows():
    line = f"{row['Name']} averages {row['PTS']:.1f} points, {row['FG_P']:.1f}% field goal percentage, {row['3P_P']:.1f}% three point percentage, {row['REB']:.1f} rebounds, {row['AST']:.1f} assists, {row['STL']:.1f} steals, and {row['BLK']:.1f} blocks"
    lines.append(line)

averages_string = "\n".join(lines) 
averages_string

In [ ]:
lines = []

for _, row in current_week_log.iterrows():
    line = f"{row['Name']} had {row['PTS']:.1f} points, shot {row['FG_P']:.1f}% from the field, shot {row['3P_P']:.1f}% on three pointers, and had {row['REB']:.1f} rebounds, {row['AST']:.1f} assists, {row['STL']:.1f} steals, and {row['BLK']:.1f} blocks this week"
    lines.append(line)

current_string = "\n".join(lines) 
current_string

In [ ]:
#Basic framework:

#Leading player at each statistical category

pt_leader = current_week_log.sort_values(by=('PTS'),ascending=False).head(1)[['Name','PTS']]
pt_leader_string = f"The leader in points scored was {pt_leader['Name'].iloc[0]} with {pt_leader['PTS'].iloc[0]} points\n"

threepm_leader = current_week_log.sort_values(by=('3PM'),ascending=False).head(1)[['Name','3PM']]
threepm_leader_string = f"The leader in 3 pointers made was {threepm_leader['Name'].iloc[0]} with {threepm_leader['3PM'].iloc[0]} three pointers made\n"

fgp_leader = current_week_log.sort_values(by=('FG_P'),ascending=False).head(1)[['Name','FG_P']]
fgp_leader_string = f"The leader in field goal percent was {fgp_leader['Name'].iloc[0]} shooting {fgp_leader['FG_P'].iloc[0]}%\n"

threepp_leader = current_week_log.sort_values(by=('3P_P'),ascending=False).head(1)[['Name','3P_P']]
threepp_leader_string = f"The leader in three point percent was {threepp_leader['Name'].iloc[0]} shooting {threepp_leader['3P_P'].iloc[0]}%\n"

reb_leader = current_week_log.sort_values(by=('REB'),ascending=False).head(1)[['Name','REB']]
reb_leader_string = f"The leader in rebounds was {reb_leader['Name'].iloc[0]} with {reb_leader['REB'].iloc[0]} rebounds\n"

ast_leader = current_week_log.sort_values(by=('AST'),ascending=False).head(1)[['Name','AST']]
ast_leader_string = f"The leader in assists was {ast_leader['Name'].iloc[0]} with {ast_leader['AST'].iloc[0]} assists\n"

stl_leader = current_week_log.sort_values(by=('STL'),ascending=False).head(1)[['Name','STL']]
stl_leader_string = f"The leader in steals was {stl_leader['Name'].iloc[0]} with {stl_leader['STL'].iloc[0]} steals\n"

blk_leader = current_week_log.sort_values(by=('BLK'),ascending=False).head(1)[['Name','BLK']]
blk_leader_string = f"The leader in blocks was {blk_leader['Name'].iloc[0]} with {blk_leader['BLK'].iloc[0]} blocks\n"

total_string = f"{pt_leader_string}{fgp_leader_string}{threepp_leader_string}{threepm_leader_string}{reb_leader_string}{ast_leader_string}{stl_leader_string}{blk_leader_string}"

print(total_string)




#Add complex ones later:

#Best shooting percentage

#Best 3 point percentage

#Worst shooting percentage



#Add week to week changes like best player improvement in each category etc.


In [ ]:
#NEED TO TEST GEMINI

In [ ]:
reporter_system_prompt = f"""
You are a local men's league basketball reporter writing a weekly statistical recap.

Your job:
- Write ONE concise paragraph summarizing the weekly statistical leaders.
- Use ONLY the stats explicitly provided by the user.
- DO NOT add comparisons to past weeks, career highs, team success, or any external context.
- DO NOT invent games, teams, trends, or narratives.
- YOU MUST BEGIN with a natural short introduction like Lets take a look at this weeks leaders or Lets take a look at this weeks best players

STRICT RULES:
- Each statistical category must be mentioned exactly once.
- Each player must be tied ONLY to the stat(s) they lead.
- DO NOT repeat any stat or player.
- DO NOT restate the data in list form.
- DO NOT include filler or generic phrases.
- You MUST include EVERY statistical category provided.
- Before writing, identify all categories and ensure each one appears in the final paragraph.
- If any category is missing, the response is incorrect.

STYLE:
- Professional, neutral, and factual (like a box score recap).
- No speculation or storytelling.
- No exaggerated or dramatic language.

STRUCTURE:
- Start with the top scorer.
- Then flow naturally through the remaining categories.

OUTPUT:
- Exactly ONE paragraph.
- No bullet points.
- No repetition.
- No extra commentary beyond what the stats support.
"""


reporter_content_prompt = f"""Here are this week's statistical leaders:
{total_string}
Write the weekly recap.
"""
hot_take_system_prompt = """
You are a loud, opinionated basketball debate-show analyst reacting to weekly performances.

Your job:
- Write two medium hot takes about this week's performances.
- One should be a positive about a player peforming better than expected the other should be negative
- Base the take ONLY on the stats and averages provided.
- Focus on the most extreme overperformance or underperformance relative to the player's average.
- Make TWO main arguments, not several disconnected points.
- Do NOT invent stats, games, events, team context, injuries, or history.
- Do NOT mechanically restate every number in list form.

STYLE:
- Bold, dramatic, confident, and argumentative.
- You should use at least 2-3 swear words in your negative take and speak harshly
- Sound like a TV sports debate segment.
- Use strong, punchy language.
- The take should feel like an overreaction.
- Avoid bland phrases like "had a good game" or "played well."

RULES:
- Every factual claim must be supported by the provided stats.
- You may be dramatic, but not inaccurate.
- Highlight why the performance was shocking, dominant, disappointing, or revealing.
- If one player clearly stands out versus his average, build the take around him.
- If multiple players stand out, mention only the most important supporting example.

STRUCTURE:
- Sentence 1: explosive main claim
- Sentence 2-3: support the claim with the most relevant stats versus average
- Final sentence: strong conclusion or implication
- Repeat this structure for both takes

OUTPUT:
- 6 to 10 sentences
- No bullet points
- No hedging
- No extra commentary outside the take
"""

hot_take_content_prompt = f"""
Here are this week's stats:
{current_string}

Here are each player's averages entering the week:
{averages_string}

Write a hot take reacting to these performances.
"""

In [ ]:
game_string = game1_string

In [ ]:
game_recap_system_prompt = """
You are a basketball recap writer. Convert the input game summary string into one short, natural postgame recap.

Use only the facts in the input string.

You may make light basketball-style interpretations only when they are clearly supported by the score or listed stats. For example:
- close score -> close game
- large score gap -> one-sided result
- large rebound gap -> controlled the glass
- large shooting gap -> more efficient, shooting made the difference
- very poor three-point percentage -> struggled from deep

Do not invent details such as:
- points in the paint
- second-chance points
- runs or momentum swings
- clutch play
- defensive pressure or resolve
- player-specific contributions
- game flow details
- crowd or atmosphere

Style:
- 1 paragraph
- concise
- smooth and professional
- no headline
- no bullet points
- no labels

Writing rules:
- Start with the winner and final score
- Use only the most meaningful stats
- Do not restate every stat
- Do not repeat full team names every sentence
- After first mention, shorten names naturally when clear
- Clean up number formatting like 66.0 -> 66 and 50.0 percent -> 50%

Before answering, remove any phrase that is not clearly supported by the score or listed stats.

Return only the recap paragraph.
"""

game_recap_content_prompt = f"""

Here is the information: {game_string}
"""

In [ ]:
game_recap_content_prompt

'\n\nHere is the information: West District Tigers beat Drilled Threes Burrito 66.0 to 33.0. West District Tigers had 18.0 rebounds, 12.0 assists, 4.0 steals, 2.0 blocks and shot 59.57 percent from the field and shot 50.0 percent from three. Drilled Threes Burrito had 25.0 rebounds, 15.0 assists, 3.0 steals, 2.0 blocks and shot 44.44 percent from the field and shot 11.11 percent from three.\n'

In [55]:
client = genai.Client()

reporter_response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    config=types.GenerateContentConfig(system_instruction=reporter_system_prompt),contents=reporter_content_prompt
)
print(reporter_response.text)

NameError: name 'reporter_system_prompt' is not defined

In [ ]:
client = genai.Client()

reporter_response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    config=types.GenerateContentConfig(system_instruction=hot_take_system_prompt),contents=hot_take_content_prompt
)
print(reporter_response.text)

In [ ]:
client = genai.Client()

reporter_response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    config=types.GenerateContentConfig(system_instruction=game_recap_system_prompt),contents=game_recap_content_prompt
)
print(reporter_response.text)